## Example 1: async_lru functionality test 1

In [ ]:
import async_lru
from async_lru import alru_cache
import asyncio

# Example 1: Basic async LRU caching
print("Example 1: Basic async LRU caching")

try:
    print("async_lru module imported successfully")
    print("Provides LRU caching for async functions")
    
    # Check available components
    attrs = [attr for attr in dir(async_lru) if not attr.startswith('_')]
    print(f"Available components: {attrs}")
    
    computed = []  # arguments the function body actually ran for
    
    # Create a simple async function with LRU cache
    @alru_cache(maxsize=3)
    async def expensive_async_operation(n):
        computed.append(n)
        print(f"Computing expensive operation for {n}")
        await asyncio.sleep(0.1)  # Simulate expensive computation
        return n * n * n  # Cube the number
    
    print("✓ Async function with LRU cache created")
    
    # Test the cached function
    async def test_basic_caching():
        # First calls - should compute
        result1 = await expensive_async_operation(2)
        result2 = await expensive_async_operation(3)
        result3 = await expensive_async_operation(4)
        
        print(f"Results: {result1}, {result2}, {result3}")
        
        # Repeat calls - should use cache
        print("Calling again (should use cache):")
        cached_result1 = await expensive_async_operation(2)
        cached_result2 = await expensive_async_operation(3)
        
        return [result1, result2, result3, cached_result1, cached_result2]
    
    results = await test_basic_caching()
    print(f"All results: {results}")
    assert results == [8, 27, 64, 8, 27], results
    assert computed == [2, 3, 4], computed  # the repeated calls were served from the cache
    info = expensive_async_operation.cache_info()
    print(f"Cache info: {info}")
    assert (info.hits, info.misses, info.currsize) == (2, 3, 3), info
    
    print("async_lru example 1 completed successfully")
    
except Exception as e:
    print(f"Runtime error: {e}")
    raise AssertionError(
        f"async_lru example (cell 1) failed: {type(e).__name__}: {e}"
    ) from e
    
print("Example 1 finished")

## Example 2: async_lru functionality test 2

In [ ]:
from async_lru import alru_cache
import asyncio
import time

# Example 2: Cache eviction and size limits
print("Example 2: Cache eviction and size limits")

try:
    # Create async function with small cache size
    fetched = []  # keys the function body actually ran for
    
    @alru_cache(maxsize=2)
    async def fetch_data(key):
        fetched.append(key)
        print(f"Fetching data for key: {key}")
        await asyncio.sleep(0.05)  # Simulate network request
        return f"data_{key}_" + str(int(time.time() * 1000) % 10000)
    
    print("✓ Async function with maxsize=2 cache created")
    
    # Test cache eviction
    async def test_cache_eviction():
        print("Testing cache eviction with size limit:")
        
        # Fill cache to capacity
        result1 = await fetch_data("A")  # Cache miss
        result2 = await fetch_data("B")  # Cache miss
        print(f"First two results: {result1[:10]}..., {result2[:10]}...")
        
        # Access cached items
        cached_a = await fetch_data("A")  # Cache hit
        cached_b = await fetch_data("B")  # Cache hit
        assert (cached_a, cached_b) == (result1, result2)  # the stored values, not new fetches
        print("Repeated calls used cache (no fetch messages)")
        
        # Add third item - should evict oldest
        print("Adding third item (should evict oldest):")
        result3 = await fetch_data("C")  # Cache miss, evicts A
        print(f"Third result: {result3[:10]}...")
        
        # Try to access A again - should be cache miss
        print("Accessing A again (should be evicted):")
        result_a_again = await fetch_data("A")  # Cache miss
        
        return [result1, result2, result3, result_a_again]
    
    eviction_results = await test_cache_eviction()
    
    # A was the least recently used key when C arrived, so it was evicted and fetched again
    assert fetched == ["A", "B", "C", "A"], fetched
    info = fetch_data.cache_info()
    print(f"Cache info: {info}")
    assert (info.hits, info.misses, info.maxsize, info.currsize) == (2, 4, 2, 2), info
    
    print("Cache eviction and size limits working")
    print("async_lru example 2 completed successfully")
    
except Exception as e:
    print(f"Runtime error: {e}")
    raise AssertionError(
        f"async_lru example (cell 3) failed: {type(e).__name__}: {e}"
    ) from e
    
print("Example 2 finished")

## Example 3: async_lru functionality test 3

In [ ]:
from async_lru import alru_cache
import asyncio

# Example 3: Cache management and clearing
print("Example 3: Cache management and clearing")

try:
    # Create async function with cache management
    made_calls = []  # (endpoint, params) pairs the function body actually ran for
    
    @alru_cache(maxsize=5)
    async def api_call(endpoint, params=None):
        made_calls.append((endpoint, params))
        print(f"Making API call to {endpoint} with params {params}")
        await asyncio.sleep(0.02)  # Simulate API delay
        
        # Simulate different responses
        if endpoint == "users":
            return {"users": [f"user_{i}" for i in range(3)]}
        elif endpoint == "posts":
            return {"posts": [f"post_{i}" for i in range(2)]}
        else:
            return {"data": f"response_for_{endpoint}"}
    
    print("✓ API call function with cache created")
    
    # Test cache management
    async def test_cache_management():
        # Make some cached calls
        result1 = await api_call("users")
        result2 = await api_call("posts") 
        result3 = await api_call("users")  # Should be cached
        
        print(f"API responses received: {len([result1, result2, result3])}")
        assert result1 == {"users": ["user_0", "user_1", "user_2"]} and result3 is result1
        assert made_calls == [("users", None), ("posts", None)], made_calls
        
        # Clearing the cache empties it and resets the statistics
        api_call.cache_clear()
        print("✓ Cache cleared")
        assert api_call.cache_info().currsize == 0
        
        # This should be a cache miss now
        result4 = await api_call("users")
        print("Called users again after cache clear")
        assert made_calls[-1] == ("users", None) and len(made_calls) == 3, made_calls
        
        # Check cache info
        info = api_call.cache_info()
        print(f"Final cache info: {info}")
        assert (info.hits, info.misses, info.currsize) == (0, 1, 1), info
        
        return [result1, result2, result3]
    
    management_results = await test_cache_management()
    
    # Test cache with different parameter combinations
    async def test_parameter_caching():
        print("\nTesting parameter-based caching:")
        
        # Same endpoint, different parameters
        # An lru cache keys on the ARGUMENTS, so every argument must be hashable — a dict raises
        # `TypeError: unhashable type: 'dict'`. Tuples of pairs are the hashable spelling of the
        # same params, and they still compare equal so the third call is a genuine cache hit.
        await api_call("search", (("q", "python"),))
        await api_call("search", (("q", "async"),))
        await api_call("search", (("q", "python"),))  # Should be cached
        
        searches = [params for endpoint, params in made_calls if endpoint == "search"]
        assert searches == [(("q", "python"),), (("q", "async"),)], searches
        print("Parameter-based caching tested")
    
    await test_parameter_caching()
    
    print("Cache management and clearing working")
    print("async_lru example 3 completed successfully")
    
except Exception as e:
    print(f"Runtime error: {e}")
    raise AssertionError(
        f"async_lru example (cell 5) failed: {type(e).__name__}: {e}"
    ) from e
    
print("Example 3 finished")

## Example 4: async_lru functionality test 4

In [ ]:
from async_lru import alru_cache
import asyncio
import hashlib

# Example 4: Practical caching for expensive async operations
print("Example 4: Practical caching for expensive async operations")

try:
    # Simulate database or external service calls
    class MockDatabase:
        def __init__(self):
            self.call_count = 0
        
        @alru_cache(maxsize=10)
        async def get_user_profile(self, user_id):
            self.call_count += 1
            print(f"DB Query #{self.call_count}: Fetching user {user_id}")
            await asyncio.sleep(0.05)  # Simulate DB query time
            
            return {
                "id": user_id,
                "name": f"User {user_id}",
                "email": f"user{user_id}@example.com",
                "settings": {"theme": "dark", "lang": "en"}
            }
        
        @alru_cache(maxsize=20)
        async def compute_analytics(self, dataset_id, metric):
            self.call_count += 1
            print(f"Analytics #{self.call_count}: Computing {metric} for {dataset_id}")
            await asyncio.sleep(0.08)  # Simulate heavy computation
            
            # Simulate different metrics
            if metric == "mean":
                return {"result": 42.5, "computed_at": "2024-01-01"}
            elif metric == "count":
                return {"result": 1000, "computed_at": "2024-01-01"}
            else:
                return {"result": 3.14, "computed_at": "2024-01-01"}
    
    print("✓ Mock database with cached methods created")
    
    # Test practical caching scenarios
    async def test_practical_caching():
        db = MockDatabase()
        
        print("Testing user profile caching:")
        # Multiple requests for same user - should cache
        profile1 = await db.get_user_profile(123)
        profile2 = await db.get_user_profile(123)  # Cached
        profile3 = await db.get_user_profile(456)  # New user
        
        print(f"User profiles fetched, DB calls: {db.call_count}")
        assert db.call_count == 2 and profile2 is profile1 and profile3["id"] == 456, db.call_count
        
        print("\nTesting analytics caching:")
        # Analytics queries - expensive operations
        analytics1 = await db.compute_analytics("dataset1", "mean")
        analytics2 = await db.compute_analytics("dataset1", "count") 
        analytics3 = await db.compute_analytics("dataset1", "mean")  # Cached
        
        print(f"Analytics computed, total DB calls: {db.call_count}")
        assert db.call_count == 4 and analytics3 is analytics1 and analytics2["result"] == 1000, db.call_count
        
        # Check cache info: each method has its own cache
        user_info = db.get_user_profile.cache_info()
        print(f"User cache info: {user_info}")
        analytics_info = db.compute_analytics.cache_info()
        print(f"Analytics cache info: {analytics_info}")
        assert (user_info.hits, user_info.misses) == (1, 2), user_info
        assert (analytics_info.hits, analytics_info.misses) == (1, 2), analytics_info
        
        return profile1, analytics1
    
    # Run practical test
    profile, analytics = await test_practical_caching()
    print(f"Sample profile: {profile['name']}")
    print(f"Sample analytics: {analytics['result']}")
    assert profile['name'] == "User 123" and analytics['result'] == 42.5
    
    print("Practical caching for expensive operations working")
    print("async_lru example 4 completed successfully")
    
except Exception as e:
    print(f"Runtime error: {e}")
    raise AssertionError(
        f"async_lru example (cell 7) failed: {type(e).__name__}: {e}"
    ) from e
    
print("Example 4 finished")

## Example 5: async_lru functionality test 5

In [ ]:
import asyncio
import async_lru
from async_lru import alru_cache
from importlib.metadata import version as distribution_version

# Example 5: async_lru package capabilities and performance optimization
print("Example 5: async_lru package capabilities and performance optimization")

try:
    print("async_lru provides LRU caching for async functions")
    
    # Package analysis
    attrs = [attr for attr in dir(async_lru) if not attr.startswith('_')]
    print(f"Available components: {attrs}")
    
    # Show version information
    print(f"async_lru version: {async_lru.__version__}")
    assert async_lru.__version__ == distribution_version("async-lru"), async_lru.__version__
    
    # Demonstrate performance benefits
    import time
    
    @alru_cache(maxsize=5)
    async def slow_computation(n):
        # Simulate expensive computation
        await asyncio.sleep(0.05)
        result = sum(i * i for i in range(n))
        return result
    
    print("✓ Performance test function created")
    
    # Performance comparison
    async def performance_test():
        print("Performance comparison:")
        
        # First call misses the cache; the next two calls hit it
        start_time = time.perf_counter()
        results = []
        for i in range(3):
            result = await slow_computation(1000)
            results.append(result)
        miss_time = time.perf_counter() - start_time
        
        # Repeat calls (cache hits)
        start_time = time.perf_counter()
        cached_results = []
        for i in range(3):
            result = await slow_computation(1000)
            cached_results.append(result)
        hit_time = time.perf_counter() - start_time
        
        print(f"Cache miss time: {miss_time:.3f}s")
        print(f"Cache hit time: {hit_time:.3f}s")
        print(f"Speedup: {miss_time / max(hit_time, 1e-9):.1f}x faster")
        assert hit_time < miss_time, (miss_time, hit_time)  # cache hits skip the sleep
        
        return results == cached_results
    
    consistent = await performance_test()
    print(f"Results consistent: {consistent}")
    assert consistent
    info = slow_computation.cache_info()
    assert (info.hits, info.misses) == (5, 1), info
    
    # Show key capabilities
    print("\nKey async_lru capabilities:")
    capabilities = [
        "🚀 LRU caching for async functions",
        "⚡ Significant performance improvements for repeated calls",
        "🔧 Configurable cache size (maxsize parameter)",
        "📊 Cache statistics and info (cache_info())",
        "🧹 Cache management (cache_clear())",
        "🔑 Parameter-aware caching (different args = different cache entries)",
        "🎯 Memory-efficient eviction of least recently used items",
        "🛡️ Thread-safe and async-safe implementation"
    ]
    
    for capability in capabilities:
        print(f"  {capability}")
    
    # Usage patterns
    print(f"\nCommon usage patterns:")
    patterns = [
        "Database query result caching",
        "API response caching for external services",
        "Expensive computation result memoization",
        "File system or network resource caching",
        "Authentication and authorization result caching",
        "Configuration and settings caching",
        "Geolocation and mapping data caching",
        "Machine learning model inference caching"
    ]
    
    for pattern in patterns:
        print(f"• {pattern}")
    
    # Performance tips
    print(f"\nPerformance optimization tips:")
    tips = [
        "Choose maxsize based on memory constraints",
        "Monitor cache hit/miss ratios with cache_info()",
        "Clear cache when data becomes stale",
        "Use immutable parameters for consistent caching",
        "Consider cache warming strategies for critical paths"
    ]
    
    for tip in tips:
        print(f"• {tip}")
    
    print(f"\nasync_lru optimizes async function performance through intelligent caching")
    print("async_lru example 5 completed successfully")
    
except Exception as e:
    print(f"Runtime error: {e}")
    raise AssertionError(
        f"async_lru example (cell 9) failed: {type(e).__name__}: {e}"
    ) from e
    
print("Example 5 finished")